# Passage EVK100 vs. Electrical Mesh — Native `!Network` Comparison

This notebook compares **Lightmatter Passage EVK100** (photonic) against an
**electrical mesh baseline** using AccelForge's built-in `!Network` component.

**Two interconnect models:**

| | Photonic (Passage EVK100) | Electrical Mesh |
|---|---|---|
| `!Network` hops energy | **0** (distance-independent) | **10 pJ/bit** per hop |
| Fixed transceiver cost | **1.9 pJ/bit** flat (via `PassageTransceiver` memory) | None |
| Cost model | Flat per-access | Scales with Manhattan distance |

**Two topologies tested:**
- **2D:** 4x4 = 16 nodes (`NODES_Z=1`)
- **3D:** 4x4x4 = 64 nodes (`NODES_Z=4`)

The photonic model uses a clever trick: the `!Network` has zero per-hop cost,
and a `PassageTransceiver` memory element adds the flat 1.9 pJ/bit energy.
This correctly models an optical crossbar where every transfer pays the same
transceiver cost regardless of distance.

## 1. Setup and Imports

In [12]:
import accelforge as af
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import os

# Paths
PROJECT_DIR = Path(".").resolve()
ARCH_DIR = PROJECT_DIR / "arch"
WORKLOAD_DIR = PROJECT_DIR.parent / "workload"
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# Architecture files
PHOTONIC_ARCH = ARCH_DIR / "passage_evk100_flat.yaml"         # Network(hops=0) + Transceiver
ELECTRICAL_ARCH = ARCH_DIR / "electrical_mesh_baseline.yaml"  # Network(hops=10 pJ/bit)
WORKLOAD_PATH = WORKLOAD_DIR / "single_matmul.yaml"

print("Architecture files:")
print(f"  Photonic:   {PHOTONIC_ARCH.name:40s} exists: {PHOTONIC_ARCH.exists()}")
print(f"  Electrical: {ELECTRICAL_ARCH.name:40s} exists: {ELECTRICAL_ARCH.exists()}")
print(f"  Workload:   {WORKLOAD_PATH.name:40s} exists: {WORKLOAD_PATH.exists()}")

Architecture files:
  Photonic:   passage_evk100_flat.yaml                 exists: True
  Electrical: electrical_mesh_baseline.yaml            exists: True
  Workload:   single_matmul.yaml                       exists: True


## 2. Jinja Variables Reference

Both arch YAMLs accept the same topology/node variables. The interconnect-specific
variables differ.

### Shared variables (defaults used by `build_spec()`)

| Variable | Default | Description |
|----------|---------|-------------|
| `NODES_X` | 4 | Nodes in X dimension |
| `NODES_Y` | 4 | Nodes in Y dimension |
| `NODES_Z` | 1 | Nodes in Z (set to 4 for 3D cube) |
| `PE_X` | 64 | PE array columns (input reuse) |
| `PE_Y` | 64 | PE array rows (output reuse) |
| `HBM_GIB` | **8** | HBM per node in GiB (matches peer's `tpu_lightmatter_cluster.yaml`) |
| `UB_MIB` | 128 | Unified buffer per node in MiB |
| `M`, `K`, `N` | 1, 4096, 16384 | Matmul dimensions |
| `BITS_PER_VALUE` | 8 | Precision |

> Note: the YAML files may declare a different default for `HBM_GIB`,
> but `build_spec()` always overrides it to **8 GiB** for an apples-to-apples
> comparison with the peer's model. All runs in this notebook use 8 GiB per node.

### Photonic-only (`passage_evk100_flat.yaml`)

| Variable | Default | Description |
|----------|---------|-------------|
| `PASSAGE_ENERGY_PER_BIT` | 1.9e-12 | Flat transceiver energy (J/bit) |
| `PASSAGE_LATENCY_PER_BIT` | 1/(8*3200e9) | Transceiver latency (s/bit) |

### Electrical-only (`electrical_mesh_baseline.yaml`)

| Variable | Default | Description |
|----------|---------|-------------|
| `ELEC_ENERGY_PER_HOP` | 10e-12 | Energy per bit **per hop** (J) |
| `ELEC_LATENCY_PER_HOP` | 5e-9 | Latency **per hop** (s) |

## 3. Helper Functions

In [13]:
def build_spec(arch_path, workload_path=WORKLOAD_PATH, **overrides):
    """
    Build an AccelForge Spec from an arch YAML + workload YAML.
    Any keyword argument is passed as a Jinja override.
    
    Examples:
        build_spec(PHOTONIC_ARCH)                                    # 2D default
        build_spec(PHOTONIC_ARCH, NODES_Z=4)                         # 3D cube
        build_spec(ELECTRICAL_ARCH, M=2048, K=4096, N=4096)          # custom workload
        build_spec(PHOTONIC_ARCH, PASSAGE_ENERGY_PER_BIT=4.6e-12)    # EVK50 energy
    """
    # Defaults match peer's tpu_lightmatter_cluster.yaml for apple-to-apple comparison
    jinja = {
        "NODES_X": 4, "NODES_Y": 4, "NODES_Z": 1,
        "PE_X": 64, "PE_Y": 64,
        "HBM_GIB": 8, "UB_MIB": 128,       # HBM_GIB=8 matches peer's default
        "M": 1, "K": 4096, "N": 16384, "BITS_PER_VALUE": 8,
    }
    jinja.update(overrides)
    
    spec = af.Spec.from_yaml(arch_path, workload_path, jinja_parse_data=jinja)
    spec.mapper.metrics = af.mapper.Metrics.ENERGY
    return spec


def run_and_report(arch_path, label="", **overrides):
    """Build spec, run mapper, return (best_mapping, energy_uJ, latency_us)."""
    spec = build_spec(arch_path, **overrides)
    mappings = spec.map_workload_to_arch(print_progress=False, print_number_of_pmappings=False)
    if len(mappings) == 0:
        print(f"  [{label}] No valid mappings found!")
        return None, None, None
    best = mappings[0]
    e_uJ = float(best.energy()) * 1e6
    l_us = float(best.latency()) * 1e6
    if label:
        print(f"  [{label:45s}]  Energy: {e_uJ:12.4f} uJ  |  Latency: {l_us:12.4f} us")
    return best, e_uJ, l_us


def component_breakdown(best_mapping):
    """Return a DataFrame of per-component energy breakdown."""
    energy_j = float(best_mapping.energy())
    per_comp = best_mapping.energy(per_component=True)
    rows = []
    for comp, e in sorted(per_comp.items(), key=lambda x: -float(x[1])):
        e_val = float(e)
        if e_val > 0:
            pct = 100 * e_val / energy_j if energy_j > 0 else 0
            rows.append({"Component": comp, "Energy (uJ)": round(e_val * 1e6, 6), "Share (%)": round(pct, 1)})
    return pd.DataFrame(rows)

In [14]:
# ---------------------------------------------------------------
# Unified technology catalog (mirrors peer's approach)
# Each entry maps to an arch file + Jinja overrides specific to that tech.
# ---------------------------------------------------------------
TECH_CATALOG = {
    "Electrical 10pJ/hop": {
        "arch": ELECTRICAL_ARCH,
        "params": {},  # uses defaults in electrical_mesh_baseline.yaml
        "kind": "electrical",
        "bw_tbps": None,
    },
    "Passage EVK50":  {"arch": PHOTONIC_ARCH, "kind": "photonic", "bw_tbps": 0.8,
                       "params": {"PASSAGE_ENERGY_PER_BIT": 4.6e-12, "PASSAGE_LATENCY_PER_BIT": 1/(8*800e9)}},
    "Passage EVK100": {"arch": PHOTONIC_ARCH, "kind": "photonic", "bw_tbps": 3.2,
                       "params": {"PASSAGE_ENERGY_PER_BIT": 1.9e-12, "PASSAGE_LATENCY_PER_BIT": 1/(8*3200e9)}},
    "Passage L200":   {"arch": PHOTONIC_ARCH, "kind": "photonic", "bw_tbps": 16,
                       "params": {"PASSAGE_ENERGY_PER_BIT": 4.5e-12, "PASSAGE_LATENCY_PER_BIT": 1/(8*16e12)}},
    "Passage L200X":  {"arch": PHOTONIC_ARCH, "kind": "photonic", "bw_tbps": 32,
                       "params": {"PASSAGE_ENERGY_PER_BIT": 4.8e-12, "PASSAGE_LATENCY_PER_BIT": 1/(8*32e12)}},
    "Passage M1000":  {"arch": PHOTONIC_ARCH, "kind": "photonic", "bw_tbps": 57,
                       "params": {"PASSAGE_ENERGY_PER_BIT": 4.0e-12, "PASSAGE_LATENCY_PER_BIT": 1/(8*57e12)}},
}

TECH_COLORS = {
    "Electrical 10pJ/hop": "#d62728",
    "Passage EVK50":       "#9edae5",
    "Passage EVK100":      "#1f77b4",
    "Passage L200":        "#2ca02c",
    "Passage L200X":       "#ff7f0e",
    "Passage M1000":       "#9467bd",
}

def run_tech(tech_name, label=None, **overrides):
    """Run a specific tech from TECH_CATALOG with extra Jinja overrides."""
    cfg = TECH_CATALOG[tech_name]
    params = {**cfg["params"], **overrides}
    return run_and_report(cfg["arch"], label or tech_name, **params)

print(f"Technology catalog: {list(TECH_CATALOG)}")

Technology catalog: ['Electrical 10pJ/hop', 'Passage EVK50', 'Passage EVK100', 'Passage L200', 'Passage L200X', 'Passage M1000']


## 4. Single Run — Photonic vs. Electrical (4x4, decode MLP)

In [15]:
print("=" * 70)
print("  4x4 mesh (16 nodes) -- decode_mlp (M=1, K=4096, N=16384)")
print("=" * 70)

best_ph, e_ph, l_ph = run_and_report(PHOTONIC_ARCH, "Photonic EVK100 (flat cost)")
best_el, e_el, l_el = run_and_report(ELECTRICAL_ARCH, "Electrical Mesh (per-hop cost)")

if e_ph and e_el:
    print(f"\n  Energy reduction:  {100*(1 - e_ph/e_el):.1f}%")
    print(f"  Latency reduction: {100*(1 - l_ph/l_el):.1f}%")

  4x4 mesh (16 nodes) -- decode_mlp (M=1, K=4096, N=16384)
  [Photonic EVK100 (flat cost)                  ]  Energy:   13659.4481 uJ  |  Latency:      66.5640 us
  [Electrical Mesh (per-hop cost)               ]  Energy:  420323.1030 uJ  |  Latency:     249.6610 us

  Energy reduction:  96.8%
  Latency reduction: 73.3%


In [16]:
# Per-component breakdown for each
if best_ph:
    print("Photonic (Passage EVK100) -- component breakdown:")
    display(component_breakdown(best_ph))

if best_el:
    print("\nElectrical Mesh -- component breakdown:")
    display(component_breakdown(best_el))

Photonic (Passage EVK100) -- component breakdown:


,Component,Energy (uJ),Share (%)
0,HBM,7550.708950,55.3
1,MainMemory,3775.354242,27.6
2,UnifiedBuffer,2327.748174,17.0
3,MAC,5.637145,0.0



Electrical Mesh -- component breakdown:


,Component,Energy (uJ),Share (%)
0,ElectricalMesh,413977.801800,98.5
1,MainMemory,3775.354242,0.9
2,UnifiedBuffer,2564.306919,0.6
3,MAC,5.637145,0.0


In [18]:
def component_breakdown(best_mapping):
    energy_j = float(best_mapping.energy())
    per_comp = best_mapping.energy(per_component=True)
    rows = []
    for comp, e in sorted(per_comp.items(), key=lambda x: -float(x[1])):
        e_val = float(e)
        pct = 100 * e_val / energy_j if energy_j > 0 else 0
        rows.append({"Component": comp, "Energy (uJ)": round(e_val * 1e6, 6), "Share (%)": round(pct, 2)})
    return pd.DataFrame(rows)

# Per-component breakdown for each
if best_ph:
    print("Photonic (Passage EVK100) -- component breakdown:")
    display(component_breakdown(best_ph))

if best_el:
    print("\nElectrical Mesh -- component breakdown:")
    display(component_breakdown(best_el))

Photonic (Passage EVK100) -- component breakdown:


,Component,Energy (uJ),Share (%)
0,HBM,7550.708950,55.28
1,MainMemory,3775.354242,27.64
2,UnifiedBuffer,2327.748174,17.04
3,MAC,5.637145,0.04
4,Register,0.000000,0.00
5,PassageOpticalFabric,0.000000,0.00
6,PassageTransceiver,0.000000,0.00



Electrical Mesh -- component breakdown:


,Component,Energy (uJ),Share (%)
0,ElectricalMesh,413977.801800,98.49
1,MainMemory,3775.354242,0.90
2,UnifiedBuffer,2564.306919,0.61
3,MAC,5.637145,0.00
4,Register,0.000000,0.00
5,HBM,0.000000,0.00


## 5. Topology Comparison — 2D (4x4) vs. 3D (4x4x4)

In [ ]:
configs = [
    {"label": "Photonic  4x4   (16 nodes)",  "arch": PHOTONIC_ARCH,    "NODES_Z": 1},
    {"label": "Photonic  4x4x4 (64 nodes)",  "arch": PHOTONIC_ARCH,    "NODES_Z": 4},
    {"label": "Electrical 4x4   (16 nodes)",  "arch": ELECTRICAL_ARCH,  "NODES_Z": 1},
    {"label": "Electrical 4x4x4 (64 nodes)",  "arch": ELECTRICAL_ARCH,  "NODES_Z": 4},
]

rows = []
for cfg in configs:
    _, e, l = run_and_report(cfg["arch"], cfg["label"], NODES_Z=cfg["NODES_Z"])
    if e is not None:
        nodes = 4 * 4 * cfg["NODES_Z"]
        rows.append({
            "Config": cfg["label"],
            "Nodes": nodes,
            "Interconnect": "Photonic" if "Photonic" in cfg["label"] else "Electrical",
            "Topology": "3D" if cfg["NODES_Z"] > 1 else "2D",
            "Energy (uJ)": round(e, 4),
            "Latency (us)": round(l, 4),
        })

df_topo = pd.DataFrame(rows)
display(df_topo)

  [Photonic  4x4   (16 nodes)                   ]  Energy:   13659.4481 uJ  |  Latency:      66.5640 us


In [ ]:
# Compute improvement for each topology
if len(df_topo) == 4:
    for topo in ["2D", "3D"]:
        ph = df_topo[(df_topo["Interconnect"] == "Photonic") & (df_topo["Topology"] == topo)]
        el = df_topo[(df_topo["Interconnect"] == "Electrical") & (df_topo["Topology"] == topo)]
        if len(ph) == 1 and len(el) == 1:
            e_red = 100 * (1 - ph["Energy (uJ)"].values[0] / el["Energy (uJ)"].values[0])
            l_red = 100 * (1 - ph["Latency (us)"].values[0] / el["Latency (us)"].values[0])
            print(f"{topo} topology -- Photonic vs. Electrical:")
            print(f"  Energy reduction:  {e_red:.1f}%")
            print(f"  Latency reduction: {l_red:.1f}%")
            print()

In [ ]:
# Bar chart comparison
if len(df_topo) >= 2:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    colors = ["#1f77b4" if "Photonic" in c else "#d62728" for c in df_topo["Config"]]

    ax1.barh(df_topo["Config"], df_topo["Energy (uJ)"], color=colors)
    ax1.set_xlabel("Energy (uJ)")
    ax1.set_title("Energy: Photonic vs. Electrical")
    ax1.invert_yaxis()

    ax2.barh(df_topo["Config"], df_topo["Latency (us)"], color=colors)
    ax2.set_xlabel("Latency (us)")
    ax2.set_title("Latency: Photonic vs. Electrical")
    ax2.invert_yaxis()

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "topo_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved to results/topo_comparison.png")

workloads = {
    "decode_mlp":        {"M": 1,    "K": 4096, "N": 16384},
    "kv_attention":      {"M": 1,    "K": 128,  "N": 8192},
    "prefill_projection":{"M": 2048, "K": 4096, "N": 4096},
    "long_ctx_prefill":  {"M": 8192, "K": 4096, "N": 4096},
    "moe_expert":        {"M": 1,    "K": 4096, "N": 4096},
    "training_wgrad":    {"M": 4096, "K": 2048, "N": 4096},
}

wl_rows = []
for wl_name, dims in workloads.items():
    for tech in TECH_CATALOG:
        _, e, l = run_tech(tech, f"{wl_name} / {tech}", **dims)
        if e is not None:
            wl_rows.append({
                "Workload": wl_name,
                "Tech": tech,
                "Energy (uJ)": round(e, 4),
                "Latency (us)": round(l, 4),
            })

df_wl = pd.DataFrame(wl_rows)
display(df_wl.pivot(index="Workload", columns="Tech", values="Energy (uJ)"))

In [ ]:
# Grouped bar chart: energy per workload, one bar per tech
fig, ax = plt.subplots(figsize=(14, 6))
pivot = df_wl.pivot(index="Workload", columns="Tech", values="Energy (uJ)")
# Reorder columns to match catalog order
pivot = pivot[[t for t in TECH_CATALOG if t in pivot.columns]]
pivot.plot(kind="bar", ax=ax, color=[TECH_COLORS[t] for t in pivot.columns],
           logy=True, width=0.8)
ax.set_ylabel("Energy (uJ, log scale)")
ax.set_title("Energy per Workload — All Interconnect Technologies")
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
ax.grid(True, axis="y", alpha=0.3)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "workload_all_tech.png", dpi=150, bbox_inches="tight")
plt.show()

# Per-workload reduction vs electrical
ref = df_wl[df_wl["Tech"] == "Electrical 10pJ/hop"].set_index("Workload")["Energy (uJ)"]
imp = df_wl.copy()
imp["vs Electrical (%)"] = imp.apply(
    lambda r: round(100 * (1 - r["Energy (uJ)"] / ref[r["Workload"]]), 1), axis=1
)
print("Energy reduction vs Electrical baseline:")
display(imp[imp["Tech"] != "Electrical 10pJ/hop"].pivot(
    index="Workload", columns="Tech", values="vs Electrical (%)"))

In [ ]:
# Compute per-workload improvement
ph_wl = df_wl[df_wl["Interconnect"] == "Photonic"].set_index("Workload")
el_wl = df_wl[df_wl["Interconnect"] == "Electrical"].set_index("Workload")

if len(ph_wl) > 0 and len(el_wl) > 0:
    imp_rows = []
    for wl in ph_wl.index:
        if wl in el_wl.index:
            e_red = 100 * (1 - ph_wl.loc[wl, "Energy (uJ)"] / el_wl.loc[wl, "Energy (uJ)"])
            l_red = 100 * (1 - ph_wl.loc[wl, "Latency (us)"] / el_wl.loc[wl, "Latency (us)"])
            imp_rows.append({
                "Workload": wl,
                "Energy Reduction (%)": round(e_red, 1),
                "Latency Reduction (%)": round(l_red, 1),
            })
    df_imp = pd.DataFrame(imp_rows)
    print("Photonic vs. Electrical -- per-workload improvement:")
    display(df_imp)

scale_rows = []
# 2D scaling: NxN grids
for dim in [2, 3, 4, 5]:
    n_nodes = dim * dim
    for tech in TECH_CATALOG:
        _, e, l = run_tech(tech, f"2D {dim}x{dim} / {tech}",
                           NODES_X=dim, NODES_Y=dim, NODES_Z=1)
        if e is not None:
            scale_rows.append({"Topology": f"{dim}x{dim}", "Nodes": n_nodes,
                               "Dim": "2D", "Tech": tech,
                               "Energy (uJ)": e, "Latency (us)": l})

# 3D scaling: NxNxN cubes
for dim in [2, 3, 4]:
    n_nodes = dim ** 3
    for tech in TECH_CATALOG:
        _, e, l = run_tech(tech, f"3D {dim}x{dim}x{dim} / {tech}",
                           NODES_X=dim, NODES_Y=dim, NODES_Z=dim)
        if e is not None:
            scale_rows.append({"Topology": f"{dim}x{dim}x{dim}", "Nodes": n_nodes,
                               "Dim": "3D", "Tech": tech,
                               "Energy (uJ)": e, "Latency (us)": l})

df_scale = pd.DataFrame(scale_rows)
display(df_scale.pivot_table(index=["Dim", "Nodes"], columns="Tech", values="Energy (uJ)"))

In [ ]:
if len(df_scale) > 2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, dim_label in zip(axes, ["2D", "3D"]):
        sub = df_scale[df_scale["Dim"] == dim_label]
        for tech in TECH_CATALOG:
            d = sub[sub["Tech"] == tech].sort_values("Nodes")
            if len(d) > 0:
                ax.plot(d["Nodes"], d["Energy (uJ)"], "-o",
                        color=TECH_COLORS[tech], label=tech, linewidth=2, markersize=7)
        ax.set_xlabel("Number of Nodes")
        ax.set_ylabel("Energy (uJ)")
        ax.set_yscale("log")
        ax.set_title(f"Energy Scaling — {dim_label} Topologies")
        ax.grid(True, alpha=0.3)
    axes[-1].legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "scaling_all_tech.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved to results/scaling_all_tech.png")

In [ ]:
# Plot scaling
if len(df_scale) > 2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, dim_label in zip(axes, ["2D", "3D"]):
        sub = df_scale[df_scale["Dim"] == dim_label]
        for inter, color, marker in [("Photonic", "#1f77b4", "o"), ("Electrical", "#d62728", "s")]:
            d = sub[sub["Interconnect"] == inter]
            if len(d) > 0:
                ax.plot(d["Nodes"], d["Energy (uJ)"], f"-{marker}",
                        color=color, label=inter, linewidth=2, markersize=8)
        ax.set_xlabel("Number of Nodes")
        ax.set_ylabel("Energy (uJ)")
        ax.set_title(f"Energy Scaling -- {dim_label} Topologies")
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "scaling_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved to results/scaling_comparison.png")

## 8. Passage Variant Sweep — Different Products

In [ ]:
# Compare different Passage products by changing the transceiver energy
passage_variants = {
    "EVK50":  {"PASSAGE_ENERGY_PER_BIT": 4.6e-12, "PASSAGE_LATENCY_PER_BIT": 1/(8*800e9)},
    "EVK100": {"PASSAGE_ENERGY_PER_BIT": 1.9e-12, "PASSAGE_LATENCY_PER_BIT": 1/(8*3200e9)},
    "L200":   {"PASSAGE_ENERGY_PER_BIT": 4.5e-12, "PASSAGE_LATENCY_PER_BIT": 1/(8*16e12)},
    "L200X":  {"PASSAGE_ENERGY_PER_BIT": 4.8e-12, "PASSAGE_LATENCY_PER_BIT": 1/(8*32e12)},
    "M1000":  {"PASSAGE_ENERGY_PER_BIT": 4.0e-12, "PASSAGE_LATENCY_PER_BIT": 1/(8*57e12)},
}

# Run electrical baseline for reference
_, e_elec, l_elec = run_and_report(ELECTRICAL_ARCH, "Electrical Baseline")

var_rows = []
for name, params in passage_variants.items():
    _, e, l = run_and_report(PHOTONIC_ARCH, f"Passage {name}", **params)
    if e is not None and e_elec:
        e_red = 100 * (1 - e / e_elec)
        var_rows.append({
            "Variant": name,
            "Energy/bit (pJ)": params["PASSAGE_ENERGY_PER_BIT"] * 1e12,
            "Energy (uJ)": round(e, 4),
            "Latency (us)": round(l, 4),
            "vs. Electrical (%)": round(e_red, 1),
        })

df_var = pd.DataFrame(var_rows)
display(df_var)

## 9. Architecture Diagrams — What's Different

### Photonic (`passage_evk100_flat.yaml`)
```
MainMemory (shared pool)                 <- HBM energy: 7.03 pJ/bit
  |-- PassageOpticalFabric (!Network)    <- hops energy: 0 (distance-free)
      |-- PassageTransceiver (!Memory)   <- FLAT 1.9 pJ/bit per access
          |-- HBM (per-node)             <- local HBM: 7.03 pJ/bit
              |-- UnifiedBuffer
                  |-- PE Array (64x64)
                      |-- Register
                          |-- MAC
```

### Electrical (`electrical_mesh_baseline.yaml`)
```
MainMemory (shared pool)                 <- HBM energy: 7.03 pJ/bit
  |-- ElectricalMesh (!Network)          <- 10 pJ/bit x HOPS (distance-dependent)
      |-- HBM (per-node, spatial)        <- local HBM: 7.03 pJ/bit
          |-- UnifiedBuffer
              |-- PE Array (64x64)
                  |-- Register
                      |-- MAC
```

The key structural difference: the photonic model has an extra level
(`PassageTransceiver`) that captures the flat optical cost, while the
electrical model lets the `!Network` handle everything via hop scaling.

## 10. Next Steps

- **Compare with peer's iterative approach:** run the same workloads through
  `scripts/lightmatter_interconnects.py` and compare the results
- **Add multicast modeling:** the flat-cost model doesn't distinguish
  unicast vs. multicast -- consider the iterative approach for that
- **Try asymmetric topologies:** e.g., `NODES_X=8, NODES_Y=2, NODES_Z=1`
- **Sweep precision:** compare 8-bit vs. 16-bit via `BITS_PER_VALUE`